## SCD Híbrido: Tipo 1 + Tipo 2

Esta dimensión usa una **estrategia híbrida** que combina ambos tipos de SCD según el atributo:

### **SCD Tipo 2** - `max_contenidos_mensuales`
* **Mantiene historial completo** de cambios en el límite de contenidos
* Cada cambio crea un nuevo registro con `valido_desde` / `valido_hasta`
* Permite analizar cómo evolucionaron los límites de cada plan a lo largo del tiempo
* **Caso de uso**: "¿Cuántos contenidos permitía el plan Premium en enero 2024?"

### **SCD Tipo 1** - `nombre_suscripcion`
* **NO mantiene historial** de cambios en el nombre
* Actualiza el nombre en TODOS los registros (históricos y actuales)
* Concepto: "Este nombre siempre fue así, solo lo corregimos"
* **Caso de uso**: Corrección de typos, rebranding, estandarización de nombres

---

## ¿Por qué esta estrategia?

**Tipo 2 para límites**: Los límites de contenido son **decisiones de negocio** que cambian en el tiempo y afectan análisis:
* Reportes de uso vs límites históricos
* Análisis de upgrades/downgrades
* Cumplimiento de contratos

**Tipo 1 para nombres**: Los nombres son **etiquetas descriptivas** que no afectan análisis:
* "Plan Premium" vs "Premium Plan" → mismo plan, diferente etiqueta
* Corregir typos sin crear versiones innecesarias
* Mantener consistencia en reportes históricos

---

## Implementación: 4 pasos

1. **Staging**: Detectar cambios en `max_contenidos_mensuales` (solo Tipo 2)
2. **Expirar**: MERGE para marcar registros modificados como `es_actual = FALSE`
3. **Insertar**: INSERT de nuevos registros y nuevas versiones de modificados
4. **Actualizar nombres**: MERGE Tipo 1 para `nombre_suscripcion` en toda la historia

In [0]:
-- ============================================
-- PASO 1: Crear tabla de staging con detección de cambios
-- ============================================
-- Detecta cambios solo en max_contenidos_mensuales (SCD Tipo 2)
-- El nombre_suscripcion se actualizará con SCD Tipo 1 en paso separado

CREATE OR REPLACE TABLE IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_suscripciones') AS
SELECT 
    s.id_suscripcion,
    s.nombre_suscripcion,
    s.max_contenidos_mensuales,
    s.fecha_creacion,
    s.fecha_actualizacion,
    CASE 
        WHEN d.id_suscripcion IS NULL THEN 'NUEVO'
        WHEN s.max_contenidos_mensuales != d.max_contenidos_mensuales THEN 'MODIFICADO'
        ELSE 'SIN_CAMBIO'
    END as accion
FROM IDENTIFIER(:catalogo_origen || '.core_negocio_' || :alumno || '.suscripciones') s
LEFT JOIN (
    SELECT * 
    FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_suscripciones')
    WHERE es_actual = TRUE
) d ON s.id_suscripcion = d.id_suscripcion;

In [0]:
-- ============================================
-- PASO 2: SCD Tipo 2 - Expirar registros que han cambiado
-- ============================================
-- Solo para cambios en max_contenidos_mensuales
-- IMPORTANTE: Usamos DATE(s.fecha_actualizacion) para valido_hasta
-- en vez de CURRENT_DATE() para que la operación sea IDEMPOTENTE.

MERGE INTO IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_suscripciones') AS target
USING IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_suscripciones') AS s
ON target.id_suscripcion = s.id_suscripcion 
   AND target.es_actual = TRUE
   AND s.accion = 'MODIFICADO'
WHEN MATCHED THEN
    UPDATE SET
        target.valido_hasta = DATE(s.fecha_actualizacion) - INTERVAL 1 DAY,
        target.es_actual = FALSE;

In [0]:
-- ============================================
-- PASO 3: SCD Tipo 2 - Insertar nuevos registros (nuevos + modificados)
-- ============================================
-- NOTA: id_dim_suscripcion se genera automáticamente (IDENTITY)
-- Usamos DATE(fecha_actualizacion) como valido_desde para idempotencia

INSERT INTO IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_suscripciones')
    (id_suscripcion, nombre_suscripcion, max_contenidos_mensuales, fecha_creacion, 
     fecha_actualizacion, valido_desde, valido_hasta, es_actual)
SELECT
    s.id_suscripcion,
    s.nombre_suscripcion,
    s.max_contenidos_mensuales,
    s.fecha_creacion,
    s.fecha_actualizacion,
    DATE(s.fecha_actualizacion) as valido_desde,
    DATE '9999-12-31' as valido_hasta,
    TRUE as es_actual
FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_suscripciones') s
WHERE s.accion IN ('NUEVO', 'MODIFICADO');

In [0]:
-- ============================================
-- PASO 4: SCD Tipo 1 - nombre_suscripcion
-- ============================================
-- Actualiza el nombre en TODOS los registros (históricos y actuales) sin crear historial
-- Concepto: "Este nombre siempre fue así, lo corregimos en toda la historia"

MERGE INTO IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_suscripciones') AS destino
USING IDENTIFIER(:catalogo_origen || '.core_negocio_' || :alumno || '.suscripciones') AS plata
ON destino.id_suscripcion = plata.id_suscripcion
WHEN MATCHED AND destino.nombre_suscripcion != plata.nombre_suscripcion THEN
    UPDATE SET
        destino.nombre_suscripcion = plata.nombre_suscripcion,
        destino.fecha_actualizacion = plata.fecha_actualizacion;

In [0]:
-- Mostrar una muestra de los datos de dim_suscripciones
-- Incluye tanto registros actuales como históricos
SELECT 
    id_dim_suscripcion,
    id_suscripcion,
    nombre_suscripcion,
    max_contenidos_mensuales,
    valido_desde,
    valido_hasta,
    es_actual,
    fecha_actualizacion
FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_suscripciones')
ORDER BY id_suscripcion, valido_desde DESC
LIMIT 20;